# Chess Expert — Train, Watch, Play & Share (Colab)

Downloads grandmaster games, trains the model (policy + value heads) on a GPU, logs everything to **TensorBoard**, lets you **play on a clickable board**, and **uploads to Hugging Face**.

**Before you run anything:**
1. `Runtime → Change runtime type →` **GPU**. A **T4** works but is slow; **L4** (best value) or **A100** is much faster.
2. In the *Clone* cell, set `REPO_URL` to your GitHub repo.

> 💡 **Dry run first.** Run the whole notebook once with `--max-games 200` (Parse) and `--epochs 2` (Train) to prove the chain works for pennies, then remove the caps.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Get the code and install dependencies

In [ ]:
REPO_URL = "https://github.com/AhPro7/chess-expert.git"  # <-- your repo

import os
if not os.path.isdir("chess-expert"):
    !git clone $REPO_URL
%cd chess-expert
!git pull
!pip -q install -r requirements.txt

## 3. Download grandmaster games
Real GM archives merged into `data/gm_games.pgn`. Edit `scripts/download_data.sh` to add players.

In [ ]:
!bash scripts/download_data.sh
!ls -lh data/gm_games.pgn

## 4. Parse PGN → training samples
Saves `positions.npy`, `moves.npy`, and **`values.npy`** (game outcomes, for the value head). Keep all GM games (`--min-elo 0`); add `--max-games 5000` for a quick pass.

In [ ]:
!python -m src.data --pgn data/gm_games.pgn --out data/samples --min-elo 0

## 5. Train (policy + value)
Logs to `runs/` for TensorBoard. Watch **val move-match** (policy) and **val value-MAE** (value; lower is better). On L4/A100 add `--amp on`. **Resume** a stopped run with `--resume models/chess_expert.resume.pt`.

In [ ]:
!python -m src.train \
  --data data/samples \
  --out models/chess_expert.pt \
  --epochs 20 \
  --batch-size 4096 \
  --lr 1e-3 \
  --channels 128 --blocks 10
# OOM? lower --batch-size (e.g. 2048). Disable image logging with --no-tb-images.

## 6. Watch training in TensorBoard
Live loss/accuracy curves **and** a self-play board filmstrip per epoch (Images tab). Run this while (or after) training.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 7. Sanity check — self-play a full game (asserts every move is legal)

In [ ]:
!python -m src.play --checkpoint models/chess_expert.pt --plies 60 --depth 2

## 8. 🎮 Play against it — clickable board
Click one of your pieces, then click where it should go. The engine **looks 2 moves ahead** (value head). `depth=3` = stronger/slower; `human_white=False` to play Black.

In [ ]:
from demo.colab_gui import play
play("models/chess_expert.pt")  # you are White

## 9. Make the demo GIF (for LinkedIn) and download it

In [ ]:
!python -m demo.make_gif --checkpoint models/chess_expert.pt --out demo/self_play.gif --plies 60 --temperature 0.6
from google.colab import files
files.download('demo/self_play.gif')

## 10. ☁️ Save the model + logs to Hugging Face
Log in (paste a token from https://huggingface.co/settings/tokens), then upload the checkpoint and TensorBoard logs to your Hub repo.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!python scripts/upload_hf.py --repo-id AhPro7/chess-expert  # <-- your HF username
# uploads chess_expert.pt + logs/ + a model card

## 11. Download the trained model locally
Also drop it into your local repo's `models/` folder (and commit it) so anyone can play on clone.

In [ ]:
from google.colab import files
files.download('models/chess_expert.pt')